**Подготовка файла 3100.csv на основе данных сводных файлов 2016-2018 годов**

- Автор: Клинкова Екатерина
- Telegram: @ekaklinkova

- `..\Data\{год}\` - папка для хранения созданнного 2100.csv для определенного года
- `..\Data\{год}\Original` - папка для хранения оригинального файла xlsx
- `..\Data\Common\` - папка для хранения итогового результата 2100.csv сводный

Пример: 
- *E:/IrkutskProject/Data/2016/Original/ф33  правл за 2016г Иркутская обл.xlsx*
- *E:/IrkutskProject/Data/2016/2100.csv*

In [4]:
# Загрузка библиотек
import pandas as pd
import numpy as np
import os
import docx
import re

In [5]:
path = 'E:/IrkutskProject/Data/'
# Список годов для обработки
years = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

# Название оригинального файла
original_names = {
    2016: 'ИО 30 2016.docx',
    2017: 'ИО 30 2017.docx',
    2018: 'ИО 30 2018.docx',
    2019: 'ИО 30 2019.docx',
    2020: 'ИО 30 2020.docx',
    2021: '30 ИО 2021.docx',
    2022: '30 ИО 2022.docx',
    2023: '30 ИО 2023.docx',
    2024: '30 ИО 2024.docx'
}

original_url = {}
for year in years:
    original_path = f'{path}{year}/Original/'
    original_url[year] = original_path + original_names[year]
    
    if not os.path.exists(original_url[year]):
        print(f"Файл {original_url[year]} не найден по указанному пути")
    else:
        print(f"Файл {original_url[year]} найден по указанному пути")

Файл E:/IrkutskProject/Data/2016/Original/ИО 30 2016.docx найден по указанному пути
Файл E:/IrkutskProject/Data/2017/Original/ИО 30 2017.docx найден по указанному пути
Файл E:/IrkutskProject/Data/2018/Original/ИО 30 2018.docx найден по указанному пути
Файл E:/IrkutskProject/Data/2019/Original/ИО 30 2019.docx найден по указанному пути
Файл E:/IrkutskProject/Data/2020/Original/ИО 30 2020.docx найден по указанному пути
Файл E:/IrkutskProject/Data/2021/Original/30 ИО 2021.docx найден по указанному пути
Файл E:/IrkutskProject/Data/2022/Original/30 ИО 2022.docx найден по указанному пути
Файл E:/IrkutskProject/Data/2023/Original/30 ИО 2023.docx найден по указанному пути
Файл E:/IrkutskProject/Data/2024/Original/30 ИО 2024.docx найден по указанному пути


In [6]:
def extract_tuberculosis_table(file_path, target_block):
    doc = docx.Document(file_path)
    target_found = False
    table_index = -1

    # Перебираем все элементы в теле документа
    for i, element in enumerate(doc.element.body):
        # Шаг 1: Ищем текст заголовка
        if not target_found and element.tag.endswith('p'):
            text = "".join(element.itertext()).strip()
            # display(text)
            if target_block.lower() in text.lower():
                target_found = True
                print(f"Текст '{target_block}' найден. Ищем следующую таблицу...")

        # Шаг 2: После нахождения текста ищем ПЕРВУЮ встречную таблицу
        if target_found and element.tag.endswith('tbl'):
            # Считаем количество таблиц до текущего элемента
            table_count = len([e for e in doc.element.body[:i] if e.tag.endswith('tbl')])
            table_index = table_count
            print(f"Таблица найдена (индекс в документе: {table_index})")
            break  # Выходим после нахождения первой таблицы

    if not target_found:
        raise ValueError(f"Текст '{target_block}' не найден в документе")
    elif table_index == -1:
        raise ValueError(f"Текст найден, но после него нет ни одной таблицы")

    # Возвращаем всю таблицу (включая все страницы)
    return doc.tables[table_index]

def table_to_dataframe(table):
    data = []
    for row in table.rows:
        current_row = []
        
        for cell in row.cells:
            text = cell.text.strip()
            current_row.append(text)
        data.append(current_row)

    # Создаём DataFrame
    if data:  # Проверяем, что таблица не пустая
        df = pd.DataFrame(data[1:], columns=data[0])
    else:
        df = pd.DataFrame()  # Пустой DataFrame, если таблица пустая
    return df

In [7]:
try:
    df_3100 = {}
    target_block = "3100"
    for year in years:
        display(year)
        table = extract_tuberculosis_table(original_url[year], target_block)
        df_3100[year] = table_to_dataframe(table)
        df_3100[year].to_csv(f'E:/IrkutskProject/Data/{year}/3100.csv', index=False, encoding='utf-8-sig')
except Exception as e:
    print(f"Произошла ошибка: {str(e)}")

2016

Текст '3100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 93)


2017

Текст '3100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 93)


2018

...

Текст '3100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 92)


2023

Текст '3100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 92)


2024

Текст '3100' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 97)


In [8]:
df_3100[2016]

,Профиль коек,№ \nстроки,"Число коек, фактически развернутых и свернутых на ремонт","Число коек, фактически развернутых и свернутых на ремонт","Число коек, фактически развернутых и свернутых на ремонт",В отчетном году,В отчетном году,В отчетном году,В отчетном году
0,Профиль коек,№ \nстроки,на конец отчетного года,из них \nрасполо-\nженных в сельской местности,средне-годовых,"поступило пациентов, всего, чел",из них сельских жителей,из общего числа поступивших (гр.6):,из общего числа поступивших (гр.6):
1,Профиль коек,№ \nстроки,на конец отчетного года,из них \nрасполо-\nженных в сельской местности,средне-годовых,"поступило пациентов, всего, чел",из них сельских жителей,детей\n0–17 лет,лиц старше трудоспо-собного возраста
2,1,2,3,4,5,6,7,8,9
3,Всего,1,21443,2513,21208,570668,154638,119849,179093
4,в том числе:\nаллергологические для взрослых,2,20,,20,622,50,,251
...,...,...,...,...,...,...,...,...,...
114,прочие койки для взрослых,76,,,,,,,
115,прочие койки для детей,77,,,,,,,
116,"Кроме того, «движение» больных новорожденных",78,X,X,X,10570,373,10570,X
117,Из общего числа (стр. 01) – платных коек,79,39,,39,1274,94,25,221


In [9]:
df_3100_filtered = {}
for year in years:
    df_3100_filtered[year] = df_3100[year][
        (df_3100[year]['Профиль коек'] == '1') | 
        (df_3100[year]['№ \nстроки'] == '57') | 
        (df_3100[year]['№ \nстроки'] == '58')
    ]

    #df_3100_filtered[year] = df_3100_filtered[year].drop(df_3100[year].columns[7:], axis=1)
    display(df_3100_filtered[year])
 

,Профиль коек,№ \nстроки,"Число коек, фактически развернутых и свернутых на ремонт","Число коек, фактически развернутых и свернутых на ремонт","Число коек, фактически развернутых и свернутых на ремонт",В отчетном году,В отчетном году,В отчетном году,В отчетном году
2,1,2,3,4,5,6,7,8,9
94,туберкулезные для взрослых,57,1413,188,1405,5927,2037,13,862
95,туберкулезные для детей,58,150,,150,527,197,527,


,Профиль коек,№ \nстроки,"Число коек, фактически развернутых и свернутых на ремонт","Число коек, фактически развернутых и свернутых на ремонт","Число коек, фактически развернутых и свернутых на ремонт",В отчетном году,В отчетном году,В отчетном году,В отчетном году
2,1,2,3,4,5,6,7,8,9
94,туберкулезные для взрослых,57,1363,185,1356,5551,2010,8,1141
95,туберкулезные для детей,58,150,,150,457,168,457,


,Профиль коек,№ \nстроки,"Число коек, фактически развернутых и свернутых на ремонт, коек","Число коек, фактически развернутых и свернутых на ремонт, коек","Число коек, фактически развернутых и свернутых на ремонт, коек",В отчетном году,В отчетном году,В отчетном году,В отчетном году
2,1,2,3,4,5,6,7,8,9
93,ортопедические для детей,57,14,,14,285,86,285,
94,туберкулезные для взрослых,58,1333,214,1334,5544,2008,18,1144


,Профиль коек,№ \nстроки,"Число коек, фактически развернутых и свернутых на ремонт","Число коек, фактически развернутых и свернутых на ремонт","Число коек, фактически развернутых и свернутых на ремонт",В отчетном году,В отчетном году,В отчетном году,В отчетном году
2,1,2,3,4,5,6,7,8,9
94,туберкулезные для взрослых,57,1331,161,1324,5094,1832,6,1091
95,туберкулезные для детей,58,90,,95,304,149,304,


,Профиль коек,№ \nстроки,"Число коек, фактически развернутых и свернутых на ремонт, коек","Число коек, фактически развернутых и свернутых на ремонт, коек","Число коек, фактически развернутых и свернутых на ремонт, коек",В отчетном году,В отчетном году,В отчетном году,В отчетном году
2,1,2,3,4,5,6,7,8,9
93,ортопедические для детей,57,7,,7,218,78,218,
94,туберкулезные для взрослых,58,1268,199,1278,4358,1649,8,962


,Профиль коек,№ \nстроки,"Число коек, фактически развернутых и свернутых на ремонт","Число коек, фактически развернутых и свернутых на ремонт","Число коек, фактически развернутых и свернутых на ремонт",В отчетном году,В отчетном году,В отчетном году,В отчетном году
2,1,2,3,4,5,6,7,8,9
94,туберкулезные для взрослых,57,1171,174,1188,4218,1417,7,860
95,туберкулезные для детей,58,90,,90,314,112,314,


,Профиль коек,№ \nстроки,"Число коек, фактически развернутых и свернутых на ремонт, коек","Число коек, фактически развернутых и свернутых на ремонт, коек","Число коек, фактически развернутых и свернутых на ремонт, коек",В отчетном году,В отчетном году,В отчетном году,В отчетном году
2,1,2,3,4,5,6,7,8,9
96,туберкулезные для взрослых,57,1185,194,1189,4378,1634,10,634
97,туберкулезные для детей,58,90,,90,353,167,353,


,Профиль коек,№ \nстроки,"Число коек, фактически развернутых и свернутых на ремонт, коек","Число коек, фактически развернутых и свернутых на ремонт, коек","Число коек, фактически развернутых и свернутых на ремонт, коек",В отчетном году,В отчетном году,В отчетном году,В отчетном году
2,1,2,3,4,5,6,7,8,9
96,туберкулезные для взрослых,57,1185,206,1189,4215,1586,11,531
97,туберкулезные для детей,58,90,,90,320,145,320,


,Профиль коек,№ \nстроки,"Число коек, фактически развернутых и свернутых на ремонт, коек","Число коек, фактически развернутых и свернутых на ремонт, коек","Число коек, фактически развернутых и свернутых на ремонт, коек",В отчетном году,В отчетном году,В отчетном году,В отчетном году
2,1,2,3,4,5,6,7,8,9
93,туберкулезные для взрослых,57,1154,179,1173,4220,1559,8,498
94,туберкулезные для детей,58,90,,90,344,176,344,


In [10]:
columns = [
    'Показатель',
    'Уточнение',
    'Год',
    'Значение',
    'Строка',
    'Графа',
    'Таблица'
]

data = []
data_2016_2018 = []

df_3100_long = pd.DataFrame(columns=columns)
df_3100_long_2016_2018 = pd.DataFrame(columns=columns)

for year in years:
    for col_number in range(4, 6):
        for row_number in range(1, df_3100_filtered[year].shape[0]):
                new_row = []
                new_row.append(df_3100_filtered[year].iloc[row_number, 0]) # Показатель
                new_row.append(df_3100_filtered[year].columns[col_number]) # Уточнение
                new_row.append(year)
                new_row.append(df_3100_filtered[year].iloc[row_number, col_number]) # Значение
                new_row.append(df_3100_filtered[year].iloc[row_number,1]) # Строка
                new_row.append(df_3100_filtered[year].iloc[0, col_number]) # Графа
                new_row.append(3100)
                data.append(new_row)
                if year <= 2018:
                   data_2016_2018.append(new_row)

df_3100_long = pd.DataFrame(data, columns=columns)
df_3100_long_2016_2018 = pd.DataFrame(data_2016_2018, columns=columns)

In [11]:
df_3100_long_2016_2018

,Показатель,Уточнение,Год,Значение,Строка,Графа,Таблица
0,туберкулезные для взрослых,"Число коек, фактически развернутых и свернутых...",2016,1405,57,5,3100
1,туберкулезные для детей,"Число коек, фактически развернутых и свернутых...",2016,150,58,5,3100
2,туберкулезные для взрослых,В отчетном году,2016,5927,57,6,3100
3,туберкулезные для детей,В отчетном году,2016,527,58,6,3100
4,туберкулезные для взрослых,"Число коек, фактически развернутых и свернутых...",2017,1356,57,5,3100
5,туберкулезные для детей,"Число коек, фактически развернутых и свернутых...",2017,150,58,5,3100
6,туберкулезные для взрослых,В отчетном году,2017,5551,57,6,3100
7,туберкулезные для детей,В отчетном году,2017,457,58,6,3100
8,ортопедические для детей,"Число коек, фактически развернутых и свернутых...",2018,14,57,5,3100
9,туберкулезные для взрослых,"Число коек, фактически развернутых и свернутых...",2018,1334,58,5,3100


In [12]:
df_3100_long_2016_2018.to_csv(f'{path}Common_2016-2018/3100_2016-2018.csv', index=False)
df_3100_long.to_csv(f'{path}Common/3100_2016-2024.csv', index=False)